# CMS PINN — Full Training

**Before running:** Runtime → Change runtime type → GPU → select **H100** (fastest) or A100

| GPU | Expected time (100+100 epochs, full 5.38M samples) |
|-----|---------------------------------------------------|
| H100 | ~15–20 min |
| A100 | ~30–40 min |
| L4   | ~50–70 min |
| T4   | ~75–100 min |

In [ ]:
# Cell 1 — Clone repo and install dependencies
!git clone https://github.com/AtincBas/pinn-cms-project.git
%cd pinn-cms-project
!pip install tables pyarrow -q

In [ ]:
# Cell 2 — Download CERN Open Data (~627 MB)
import os, time
hdf = 'TTbar_PU50_pixelTracksDoublets_0_final.h5'
if os.path.exists(hdf):
    print(f'Already present: {os.path.getsize(hdf)/1e6:.0f} MB')
else:
    t0 = time.time()
    !wget -q --show-progress \
      'http://opendata.cern.ch/eos/opendata/cms/datascience/CNNPixelSeedsProducerTool/TTbar_13TeV_PU50_PixelSeeds/TTbar_PU50_pixelTracksDoublets_0_final.h5'
    print(f'Done in {time.time()-t0:.0f}s  ({os.path.getsize(hdf)/1e6:.0f} MB)')

In [ ]:
# Cell 3 — Verify GPU
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'Device  : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 4 — Train  (full dataset, 100 + 100 epochs)
#
# Outputs written to current directory:
#   df_test.parquet    — test-set predictions  (needed for analysis.py)
#   history_pinn.csv   — per-epoch PINN loss breakdown
#   history_pure.csv   — per-epoch Pure NN loss

!python cms_pinn.py \
    --epochs_pinn  100 \
    --epochs_pure  100 \
    --batch_size   512 \
    --lambda_pde   0.02 \
    --output_dir   .

In [ ]:
# Cell 5 — Quick results check
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_parquet('df_test.parquet')
print(f'Test set: {len(df):,} rows\n')
print(f'{"Model":<10} {"phi_MSE":>9} {"phi_R2":>8} {"eta_MSE":>9} {"eta_R2":>8}')
print('-' * 48)
for model, pc, ec in [('PINN','phi_pinn','eta_pinn'),('Pure NN','phi_pure','eta_pure')]:
    print(f'{model:<10} {mean_squared_error(df.phi_true,df[pc]):>9.4f} '
          f'{r2_score(df.phi_true,df[pc]):>8.4f} '
          f'{mean_squared_error(df.eta_true,df[ec]):>9.4f} '
          f'{r2_score(df.eta_true,df[ec]):>8.4f}')

In [ ]:
# Cell 6 — Download output files to your computer
from google.colab import files
import os
for f in ['df_test.parquet', 'history_pinn.csv', 'history_pure.csv']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}  ({os.path.getsize(f)/1e6:.1f} MB)')
    else:
        print(f'NOT FOUND: {f}')